In [ ]:
# ==================== VULNERABILITY COUNT MODELS ====================
library(fixest)
library(dplyr)
library(tidyr)
library(lubridate)

# ---- 1. Dichotomization functions -------------------------
bin_zero_pos <- function(x) {
  factor(if_else(x < 0.5, "zero", "positive"),
         levels = c("zero", "positive"))
}
bin_ten_less <- function(x) {
  factor(if_else(x > 9.5, "ten", "below_ten"),
         levels = c("below_ten", "ten"))
}
binnings <- list(zero_pos = bin_zero_pos, ten_less = bin_ten_less)

chosen_binning <- c(
  Maintained_score           = "continuous",
  Code_Review_score          = "continuous",
  Pinned_Dependencies_score  = "zero_pos",
  Security_Policy_score      = "zero_pos",
  Token_Permissions_score    = "zero_pos",
  DependUpTool_score         = "zero_pos",
  Binary_Artifacts_score     = "ten_less",
  Dangerous_Workflow_score   = "ten_less",
  Contrib_score              = "ten_less"
)

controls <- "log1p(downloads_7_day_total) + log1p(dependents) + log1p(loc) + version_age_days + log1p(Release_count)"
fe_part  <- "package_name + year_month"

# ---- 2. Function to run one model per check -----------------------------
run_univariate_model <- function(panel, check_name, binning_type, outcome_var) {
  panel_check <- panel
  if (binning_type == "continuous") {
    predictor_term <- check_name
  } else {
    bin_col <- paste0(check_name, "_bin")
    f <- binnings[[binning_type]]
    panel_check[[bin_col]] <- f(panel_check[[check_name]])
    predictor_term <- bin_col
  }
  fml_str <- paste(
    outcome_var, "~", predictor_term, "+", controls,
    "|", fe_part
  )
  model <- feglm(as.formula(fml_str), data = panel_check,
                  family = "poisson", cluster = ~package_name)
  list(model = model, n_obs = nobs(model), check = check_name, outcome = outcome_var)
}

# ---- 3. Run for every check ------------------------------------------------
panel_vuln <- read.csv("../data/monthly_panel_vuln.csv") 

vuln_results <- lapply(names(chosen_binning), function(check_name) {
  run_univariate_model(panel_vuln, check_name, chosen_binning[[check_name]], "Vulnerability_count")
})
names(vuln_results) <- names(chosen_binning)

# ---- 4. Print summary() for each model -------------------------------------
cat("=== Vulnerability Count: Univariate Model Summaries ===\n\n")
for (check_name in names(vuln_results)) {
  cat("----", check_name, "----\n")
  print(summary(vuln_results[[check_name]]$model))
  cat("\n")
}

In [ ]:
# ==================== MTTU MODELS ====================
library(fixest)
library(dplyr)
library(tidyr)
library(lubridate)

# ---- 1. Dichotomization functions -------------------------
bin_zero_pos <- function(x) {
  factor(if_else(x < 0.5, "zero", "positive"),
         levels = c("zero", "positive"))
}
bin_ten_less <- function(x) {
  factor(if_else(x > 9.5, "ten", "below_ten"),
         levels = c("below_ten", "ten"))
}
binnings <- list(zero_pos = bin_zero_pos, ten_less = bin_ten_less)

chosen_binning <- c(
  Maintained_score           = "continuous",
  Code_Review_score          = "continuous",
  Pinned_Dependencies_score  = "zero_pos",
  Security_Policy_score      = "zero_pos",
  Token_Permissions_score    = "zero_pos",
  DependUpTool_score         = "zero_pos",
  Binary_Artifacts_score     = "ten_less",
  Dangerous_Workflow_score   = "ten_less",
  Contrib_score              = "ten_less"
)

controls <- "log1p(downloads_7_day_total) + log1p(dependents) + log1p(loc) + version_age_days + log1p(Release_count)"
fe_part  <- "package_name + year_month"

# ---- 2. Function to run one model per check ---------------------------------
run_univariate_model <- function(panel, check_name, binning_type, outcome_var) {
  panel_check <- panel
  if (binning_type == "continuous") {
    predictor_term <- check_name
  } else {
    bin_col <- paste0(check_name, "_bin")
    f <- binnings[[binning_type]]
    panel_check[[bin_col]] <- f(panel_check[[check_name]])
    predictor_term <- bin_col
  }
  fml_str <- paste(
    outcome_var, "~", predictor_term, "+", controls,
    "|", fe_part
  )
  model <- feglm(as.formula(fml_str), data = panel_check,
                  family = "poisson", cluster = ~package_name)
  list(model = model, n_obs = nobs(model), check = check_name, outcome = outcome_var)
}

# ---- 3. Run for every check ------------------------------------------------
panel_mttu <- read.csv("../data/monthly_panel_mttu.csv")  # update path as needed

mttu_results <- lapply(names(chosen_binning), function(check_name) {
  run_univariate_model(panel_mttu, check_name, chosen_binning[[check_name]], "MTTU")
})
names(mttu_results) <- names(chosen_binning)

# ---- 4. Print summary() for each model -------------------------------------
cat("=== MTTU: Univariate Model Summaries ===\n\n")
for (check_name in names(mttu_results)) {
  cat("----", check_name, "----\n")
  print(summary(mttu_results[[check_name]]$model))
  cat("\n")
}